# Бизнес-алёрты на истории ПРАЙМ

День событий сравниваем с медианой предыдущих восьми таких же дней недели. Для оценки нужны как минимум четыре сопоставимых дня. Открытый инцидент остаётся открытым до подтверждённой нормы. Меняя границы прогона, можно проверить другие даты. Пороги предварительные.

In [ ]:
from pathlib import Path
import sys
from datetime import date, timedelta

import pandas as pd

notebooks_dir = Path.cwd() / "notebooks"
if notebooks_dir.is_dir() and str(notebooks_dir) not in sys.path:
    sys.path.insert(0, str(notebooks_dir))

from prime_analytics_data import load_daily

from prime.alerts import detect_anomalies, update_incidents

In [ ]:
start = date(2026, 3, 1)
end = date(2026, 8, 31)
daily, ended_by_channel = load_daily(start - timedelta(weeks=8), end, subscriptions=True)
top_channels = ended_by_channel.drop_duplicates("day").set_index("day")["channel"]

incidents = set()
rows = []
for day in pd.date_range(start, end - timedelta(days=1)):
    assessments = detect_anomalies(daily, day)
    incidents, new = update_incidents(assessments, incidents)
    rows.append({
        "дата отчёта": (day + pd.Timedelta(days=1)).date(),
        "дата событий": day.date(),
        "завершения": daily.at[day, "ended_subscriptions"],
        "ведущий канал завершений": top_channels.get(day) if "ended_subscriptions" in new else None,
        "статус выручки": assessments["revenue"]["status"],
        "статус среднего платежа": assessments["average_payment"]["status"],
        "статус завершений": assessments["ended_subscriptions"]["status"],
        "новые уведомления": ", ".join(new),
    })

results = pd.DataFrame(rows).set_index("дата отчёта")
control = [date(2026, 3, 2), date(2026, 4, 15), date(2026, 4, 16),
           date(2026, 6, 10), date(2026, 8, 23), date(2026, 8, 24), date(2026, 8, 31)]
display(results.loc[results.index.intersection(control)])
display(results.loc[results["новые уведомления"].ne("")])

Полноту платежей оцениваем по покрытию начала и конца суток и наличию сессий во всём источнике, сами метрики считаются по назначенному клиентскому сегменту. Пропуски платежей исключаются из сравнения. Для завершений у нового инцидента показан канал с наибольшим числом завершений за день. Порог и число уведомлений можно проверить по таблице `results`. Состояние инцидентов здесь действует в рамках одного прогона истории.